## 1. Install and Import Libraries
Install Kaggle API if needed and import necessary libraries.

In [ ]:
import pandas as pd
import os
from kaggle.api.kaggle_api_extended import KaggleApi

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 2. Download Datasets from Kaggle
Authenticate Kaggle API and download all required datasets automatically.

In [ ]:
api = KaggleApi()
api.authenticate()

datasets = [
    "rehan497/historic-football-match-outcomes-18722022",
    "patateriedata/all-international-football-results",
    "martj42/international-football-results-from-1872-to-2017",
]

for url in datasets:
    dest = os.path.join(RAW_DIR, url.split("/")[1])
    os.makedirs(dest, exist_ok=True)
    api.dataset_download_files(url, path=dest, unzip=True)

## 3. Load CSV Files
We define a helper function to load multiple CSVs at once.

In [ ]:
def load_csvs(paths):
    return [pd.read_csv(os.path.join(RAW_DIR, p)) for p in paths]


paths = [
    "all-international-football-results/all_matches.csv",
    "historic-football-match-outcomes-18722022/decision.csv.csv",
    "international-football-results-from-1872-to-2017/results.csv",
    "historic-football-match-outcomes-18722022/FIFA Results.csv",
    "international-football-results-from-1872-to-2017/goalscorers.csv",
    "historic-football-match-outcomes-18722022/penality kick.csv.csv",
    "international-football-results-from-1872-to-2017/shootouts.csv",
    "all-international-football-results/countries_names.csv",
    "international-football-results-from-1872-to-2017/former_names.csv",
]

(
    all_matches,
    decision,
    results,
    FIFA_Results,
    goalscorers,
    penalty_kick,
    shootouts,
    countries_names,
    former_names_csv,
) = load_csvs(paths)

## 4. Create Country Name Mapping
Combine the countries and former names datasets into a single mapping dictionary.

In [ ]:
former_names = pd.concat(
    [
        countries_names.iloc[:, :2].rename(
            columns={
                countries_names.columns[0]: "former",
                countries_names.columns[1]: "current",
            }
        ),
        former_names_csv.iloc[:, [1, 0]].rename(
            columns={
                former_names_csv.columns[1]: "former",
                former_names_csv.columns[0]: "current",
            }
        ),
    ],
    ignore_index=True,
).drop_duplicates()

mapping = dict(zip(former_names["former"], former_names["current"]))

## 5. Apply Country Name Mapping
Replace all occurrences of former country names in all relevant dataframes automatically.

In [ ]:
for df in [all_matches, decision, results]:
    df.replace(mapping, inplace=True)

## 6. Standardize Date Columns
Convert date columns to a uniform `YYYY-MM-DD` format in penalty and shootout datasets.

In [ ]:
for df in [penalty_kick, shootouts]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["date"] = df["date"].dt.strftime("%Y-%m-%d")

## 7. Fix Column Typo in Penalty Dataset

In [ ]:
penalty_kick.rename(columns={"apponent_team": "away_team"}, inplace=True)

## 8. Merge Matches

In [ ]:
matches = all_matches.set_index(["home_team", "away_team", "date"])
matches = matches.combine_first(decision.set_index(["home_team", "away_team", "date"]))
matches = matches.combine_first(results.set_index(["home_team", "away_team", "date"]))
matches = matches.reset_index()

## 9. Merge Scorers and Penalties

In [ ]:
scorers = pd.concat([FIFA_Results, goalscorers], ignore_index=True).drop_duplicates()
penalties = pd.concat([penalty_kick, shootouts], ignore_index=True).drop_duplicates()

## 10. Save Processed CSVs

In [ ]:
matches.to_csv(f"{PROCESSED_DIR}/matches.csv", index=False)
scorers.to_csv(f"{PROCESSED_DIR}/scorers.csv", index=False)
penalties.to_csv(f"{PROCESSED_DIR}/shootouts.csv", index=False)
former_names.to_csv(f"{PROCESSED_DIR}/former_names.csv", index=False)